# Nettoyage de la base d'apprentissage

In [37]:
import numpy as np
import pandas as pd

# On charge la base d'apprentissage
df = pd.read_csv("../data_finale/base_apprentissage.csv")

## Traitement des doublons

In [38]:
# Doublons joueurs + saison + club
print("\nVérification des doublons par triplet [Joueur + Saison + Club] :")

# Identification des colonnes clés
col_joueur = "player" if "player" in df.columns else None
col_saison = "season" if "season" in df.columns else None
col_club = "team" if "team" in df.columns else None

if col_joueur and col_saison and col_club:
    # On cherche les doublons en incluant le club dans le subset
    nb_doublons_stricts = df.duplicated(
        subset=[col_joueur, col_saison, col_club]
    ).sum()

    if nb_doublons_stricts > 0:
        print(
            f"   Attention : {nb_doublons_stricts} lignes sont des doublons stricts (Même joueur, même saison, même club) !"
        )
        print("   Exemple de lignes concernées :")
        print(
            df[
                df.duplicated(
                    subset=[col_joueur, col_saison, col_club], keep=False
                )
            ][[col_joueur, col_saison, col_club]].head(6)
        )
    else:
        print(
            "   Parfait ! Aucune ligne en doublon pour un même joueur, une même saison et un même club."
        )
        print(
            "   Cela prouve que vos 1656 doublons précédents étaient uniquement dus aux transferts de mi-saison (mercato)."
        )
else:
    print(
        "   • Impossible de vérifier : les colonnes 'player', 'season' ou 'team' sont manquantes."
    )


Vérification des doublons par triplet [Joueur + Saison + Club] :
   Parfait ! Aucune ligne en doublon pour un même joueur, une même saison et un même club.
   Cela prouve que vos 1656 doublons précédents étaient uniquement dus aux transferts de mi-saison (mercato).


In [39]:
print(f"Format initial de la base : {df.shape}")

# Ajouter un compteur de valeurs manquantes
df["nb_manquants_ligne"] = df.isnull().sum(axis=1)


# On trie par Joueur, Saison, Club, ET par le nombre de manquants CROISSANT
# Ainsi, pour un même joueur/saison/club, la ligne avec le MOINS de NaN sera en haut
df_tri_tech = df.sort_values(
    by=["player", "season", "team", "nb_manquants_ligne"],
    ascending=[True, True, True, True],
)

# On supprime en gardant la première (donc celle qui a le moins de manquants)
df_sans_doublons_tech = df_tri_tech.drop_duplicates(
    subset=["player", "season", "team"], keep="first"
)

print(
    f"Format après suppression des doublons techniques (ligne la plus complète gardée) : {df_sans_doublons_tech.shape}"
)

df_sans_doublons_tech.to_csv("../data_finale/base_apprentissage.csv", index=False)

Format initial de la base : (16335, 130)
Format après suppression des doublons techniques (ligne la plus complète gardée) : (16335, 130)


In [40]:
# Doublons joueurs + saison (liés au mercato)

# On recharge le nouveau dataset
df = pd.read_csv("../data_finale/base_apprentissage.csv")

print("\nVérification des doublons par couple [Joueur + Saison] liés au mercato :")
# Identification automatique des colonnes de clé
col_joueur = "player" if "player" in df.columns else None
col_saison = "season" if "season" in df.columns else None

if col_joueur and col_saison:
    nb_doublons = df.duplicated(subset=[col_joueur, col_saison]).sum()
    if nb_doublons > 0:
        print(
            f"   Attention : {nb_doublons} lignes sont des doublons stricts pour le même joueur lors de la même saison !"
        )
        print("   Exemple de lignes concernées :")
        print(
            df[df.duplicated(subset=[col_joueur, col_saison], keep=False)][
                [col_joueur, col_saison]
            ].head(4)
        )
else:
    print(
        "   • Impossible de vérifier : les colonnes 'player' ou 'season' sont manquantes."
    )


Vérification des doublons par couple [Joueur + Saison] liés au mercato :
   Attention : 869 lignes sont des doublons stricts pour le même joueur lors de la même saison !
   Exemple de lignes concernées :
          player  season
40  Aarón Martín    2021
41  Aarón Martín    2021
49  Abakar Sylla    2526
50  Abakar Sylla    2526


In [ ]:
# TODO : gérer les doublons de mercato

## Traitement des variables avec beaucoup de valeurs manquantes

In [41]:
# Variables avec beaucoup de valeurs manquantes
print("Colonnes avec plus de 30% de valeurs manquantes :")
taux_manquants = df.isnull().mean()
colonnes_vides = taux_manquants[taux_manquants > 0.30].sort_values(
    ascending=False
)
if not colonnes_vides.empty:
    for col, tx in colonnes_vides.items():
        print(f"   • {col} : {tx*100:.1f}% de valeurs manquantes")
else:
    print("   • Aucune colonne ne dépasse 30% de lignes vides.")

Colonnes avec plus de 30% de valeurs manquantes :
   • Performance_PKwon : 100.0% de valeurs manquantes
   • Performance_PKcon : 100.0% de valeurs manquantes
   • Penalty Kicks_Save% : 94.8% de valeurs manquantes
   • Performance_CS% : 93.1% de valeurs manquantes
   • Performance_Save% : 92.9% de valeurs manquantes
   • Performance_GA : 92.7% de valeurs manquantes
   • Performance_W : 92.7% de valeurs manquantes
   • Performance_Saves : 92.7% de valeurs manquantes
   • Performance_SoTA : 92.7% de valeurs manquantes
   • Performance_GA90 : 92.7% de valeurs manquantes
   • Penalty Kicks_PKatt : 92.7% de valeurs manquantes
   • Performance_D : 92.7% de valeurs manquantes
   • Performance_L : 92.7% de valeurs manquantes
   • Performance_CS : 92.7% de valeurs manquantes
   • Penalty Kicks_PKm : 92.7% de valeurs manquantes
   • Penalty Kicks_PKsv : 92.7% de valeurs manquantes
   • Penalty Kicks_PKA : 92.7% de valeurs manquantes
   • xg : 49.1% de valeurs manquantes
   • xa : 49.1% de valeurs

On sépare les joueurs de champs des gardiens car on remarque que les variables ayant un fort taux de valeurs manquantes concerne les gardiens.

In [42]:
# On sépare les gardiens des joueurs de champ

# On s'assure que le poste 'pos' est bien au format texte pour filtrer
df["pos"] = df["pos"].astype(str)

# Uniquement les Gardiens de but (GK)
df_gardiens = df[df["pos"].str.contains("GK|Gardien", na=False)].copy()

# Tous les autres joueurs de champ (Attaquants, Milieux, Défenseurs)
df_champs = df[~df["pos"].str.contains("GK|Gardien", na=False)].copy()

In [43]:
# Variables avec beaucoup de valeurs manquantes
print("Colonnes avec plus de 80% de valeurs manquantes :")
taux_manquants = df_gardiens.isnull().mean()

# On isole la série des colonnes vides
colonnes_vides_serie = taux_manquants[taux_manquants > 0.80].sort_values(
    ascending=False
)

if not colonnes_vides_serie.empty:
    # Affichage pour le suivi
    for col, tx in colonnes_vides_serie.items():
        print(f"   • {col} : {tx*100:.1f}% de valeurs manquantes")

    # On extrait la liste des noms des colonnes
    listes_cols_a_supprimer = colonnes_vides_serie.index.tolist()

    # On applique la suppression directement sur df_gardiens
    df_gardiens = df_gardiens.drop(columns=listes_cols_a_supprimer, errors="ignore")
    print(
        f"\n Succès : {len(listes_cols_a_supprimer)} colonnes ont été supprimées de la base gardiens."
    )

else:
    print("   • Aucune colonne ne dépasse 80% de lignes vides.")

Colonnes avec plus de 80% de valeurs manquantes :
   • Performance_PKwon : 100.0% de valeurs manquantes
   • Performance_PKcon : 100.0% de valeurs manquantes
   • Standard_G/SoT : 98.9% de valeurs manquantes
   • Standard_SoT% : 96.0% de valeurs manquantes
   • Standard_G/Sh : 96.0% de valeurs manquantes
   • Subs_Mn/Sub : 83.0% de valeurs manquantes

 Succès : 6 colonnes ont été supprimées de la base gardiens.


In [44]:
# Variables avec beaucoup de valeurs manquantes
print("Colonnes avec plus de 80% de valeurs manquantes :")
taux_manquants = df_champs.isnull().mean()

# On isole la série des colonnes vides
colonnes_vides_serie = taux_manquants[taux_manquants > 0.80].sort_values(
    ascending=False
)

if not colonnes_vides_serie.empty:
    # Affichage pour le suivi
    for col, tx in colonnes_vides_serie.items():
        print(f"   • {col} : {tx*100:.1f}% de valeurs manquantes")

    # On extrait la liste des noms des colonnes
    listes_cols_a_supprimer = colonnes_vides_serie.index.tolist()

    # On applique la suppression directement sur df_champs
    df_champs = df_champs.drop(columns=listes_cols_a_supprimer, errors="ignore")
    print(
        f"\n Succès : {len(listes_cols_a_supprimer)} colonnes ont été supprimées de la base champs."
    )

else:
    print("   • Aucune colonne ne dépasse 80% de lignes vides.")

Colonnes avec plus de 80% de valeurs manquantes :
   • Performance_Save% : 100.0% de valeurs manquantes
   • Performance_PKcon : 100.0% de valeurs manquantes
   • Penalty Kicks_Save% : 100.0% de valeurs manquantes
   • Performance_PKwon : 100.0% de valeurs manquantes
   • Performance_GA : 100.0% de valeurs manquantes
   • Performance_W : 100.0% de valeurs manquantes
   • Performance_GA90 : 100.0% de valeurs manquantes
   • Performance_SoTA : 100.0% de valeurs manquantes
   • Performance_Saves : 100.0% de valeurs manquantes
   • Performance_CS : 100.0% de valeurs manquantes
   • Performance_L : 100.0% de valeurs manquantes
   • Performance_D : 100.0% de valeurs manquantes
   • Performance_CS% : 100.0% de valeurs manquantes
   • Penalty Kicks_PKsv : 100.0% de valeurs manquantes
   • Penalty Kicks_PKA : 100.0% de valeurs manquantes
   • Penalty Kicks_PKatt : 100.0% de valeurs manquantes
   • Penalty Kicks_PKm : 100.0% de valeurs manquantes

 Succès : 17 colonnes ont été supprimées de la b

## Encodage de variables

In [ ]:
# Détermination des variables catégorielles potentiellement à encoder

print("\nListe des variables catégorielles :")
cols_cat = df_gardiens.select_dtypes(include=["object"]).columns.tolist()
for col in cols_cat:
    nb_uniques = df_gardiens[col].nunique()
    print(f"   • {col} ({nb_uniques} modalités uniques)")


Liste des variables catégorielles :
   • league (5 modalités uniques)
   • team (136 modalités uniques)
   • player (430 modalités uniques)
   • nation (65 modalités uniques)
   • pos (1 modalités uniques)
   • age (179 modalités uniques)
   • join_key (430 modalités uniques)
   • match_method (5 modalités uniques)
   • date (95 modalités uniques)
   • date_of_birth (353 modalités uniques)
   • name (358 modalités uniques)
   • tm_join_key (358 modalités uniques)
   • tm_join_key_full (358 modalités uniques)


C:\Users\LouisHarle\AppData\Local\Temp\ipykernel_5940\4063206680.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cols_cat = df_gardiens.select_dtypes(include=["object"]).columns.tolist()


In [ ]:
# Détermination des variables catégorielles potentiellement à encoder

print("\nListe des variables catégorielles :")
cols_cat = df_champs.select_dtypes(include=["object"]).columns.tolist()
for col in cols_cat:
    nb_uniques = df_champs[col].nunique()
    print(f"   • {col} ({nb_uniques} modalités uniques)")


Liste des variables catégorielles :
   • league (5 modalités uniques)
   • team (137 modalités uniques)
   • player (5086 modalités uniques)
   • nation (129 modalités uniques)
   • pos (9 modalités uniques)
   • age (1653 modalités uniques)
   • join_key (5085 modalités uniques)
   • match_method (15 modalités uniques)
   • date (244 modalités uniques)
   • date_of_birth (3067 modalités uniques)
   • name (4194 modalités uniques)
   • tm_join_key (4193 modalités uniques)
   • tm_join_key_full (4193 modalités uniques)


C:\Users\LouisHarle\AppData\Local\Temp\ipykernel_5940\1128843132.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cols_cat = df_champs.select_dtypes(include=["object"]).columns.tolist()


## Traitement des outliers

In [48]:
# Identification des outliers

print("\nDétection des Outliers (Méthode de l'Écart Interquartile - IQR) :")
# On nettoie rapidement l'âge pour éviter les bugs de détection
df_gardiens["age"] = (
    df_gardiens["age"].astype(str).str.split("-").str[0].apply(pd.to_numeric, errors="coerce")
)
df_num = df_gardiens.select_dtypes(include=[np.number])

outliers_count = {}
for col in df_num.columns:
    Q1 = df_num[col].quantile(0.25)
    Q3 = df_num[col].quantile(0.75)
    IQR = Q3 - Q1
    borne_inf = Q1 - 1.5 * IQR
    borne_sup = Q3 + 1.5 * IQR

    # Compter le nombre de lignes hors limites
    nb_outliers = ((df_num[col] < borne_inf) | (df_num[col] > borne_sup)).sum()
    if nb_outliers > 0:
        outliers_count[col] = nb_outliers

# Tri pour afficher les colonnes avec le plus d'outliers (Top 10)
outliers_tries = sorted(outliers_count.items(), key=lambda x: x[1], reverse=True)
for col, nb in outliers_tries[:10]:
    print(f"   • {col} : {nb} valeurs extrêmes détectées")


Détection des Outliers (Méthode de l'Écart Interquartile - IQR) :
   • Performance_TklW : 264 valeurs extrêmes détectées
   • injury_minor_unknown_count : 264 valeurs extrêmes détectées
   • injury_minor_unknown_nb_d : 264 valeurs extrêmes détectées
   • injury_minor_unknown : 264 valeurs extrêmes détectées
   • Playing Time_Mn/MP : 247 valeurs extrêmes détectées
   • injury_minor_unknown_nb_m : 241 valeurs extrêmes détectées
   • Subs_Subs : 202 valeurs extrêmes détectées
   • Starts_Mn/Start : 165 valeurs extrêmes détectées
   • Penalty Kicks_PKm : 164 valeurs extrêmes détectées
   • injury_days_total : 160 valeurs extrêmes détectées


In [49]:
# Identification des outliers

print("\nDétection des Outliers (Méthode de l'Écart Interquartile - IQR) :")
# On nettoie rapidement l'âge pour éviter les bugs de détection
df_champs["age"] = (
    df_champs["age"].astype(str).str.split("-").str[0].apply(pd.to_numeric, errors="coerce")
)
df_num = df_champs.select_dtypes(include=[np.number])

outliers_count = {}
for col in df_num.columns:
    Q1 = df_num[col].quantile(0.25)
    Q3 = df_num[col].quantile(0.75)
    IQR = Q3 - Q1
    borne_inf = Q1 - 1.5 * IQR
    borne_sup = Q3 + 1.5 * IQR

    # Compter le nombre de lignes hors limites
    nb_outliers = ((df_num[col] < borne_inf) | (df_num[col] > borne_sup)).sum()
    if nb_outliers > 0:
        outliers_count[col] = nb_outliers

# Tri pour afficher les colonnes avec le plus d'outliers (Top 10)
outliers_tries = sorted(outliers_count.items(), key=lambda x: x[1], reverse=True)
for col, nb in outliers_tries[:10]:
    print(f"   • {col} : {nb} valeurs extrêmes détectées")


Détection des Outliers (Méthode de l'Écart Interquartile - IQR) :
   • injury_musculaire_count : 3229 valeurs extrêmes détectées
   • injury_musculaire_nb_d : 3229 valeurs extrêmes détectées
   • injury_musculaire : 3229 valeurs extrêmes détectées
   • injury_musculaire_nb_m : 3103 valeurs extrêmes détectées
   • injury_minor_unknown_nb_d : 2491 valeurs extrêmes détectées
   • injury_minor_unknown_nb_m : 2304 valeurs extrêmes détectées
   • Performance_CrdR : 1747 valeurs extrêmes détectées
   • Team Success_+/- : 1528 valeurs extrêmes détectées
   • Performance_Gls : 1492 valeurs extrêmes détectées
   • Standard_Gls : 1492 valeurs extrêmes détectées
